# The Thin Line Between Power and Fire: What Makes a Battery Separator Safe?

Every lithium-ion battery has a thin polymer membrane — the separator — preventing the cathode and anode from short-circuiting. When separators fail, batteries catch fire. In this project, we build a data-driven model to predict separator safety from polymer material properties, then screen untested polymers as potential separator candidates.

## Part 1: Data Collection

### 1a. Scraping Celgard Separator Product Data

Celgard is the largest manufacturer of battery separators. We scrape their product page to get links to all product datasheets.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
# Scrape the Celgard product data page
url = "https://www.celgard.com/product-data"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

print(f"Status code: {response.status_code}")

In [ ]:
# Find all product entries on the page
products = soup.find_all("div", attrs={"class": "ds-item"})
print(f"Found {len(products)} products")

In [ ]:
# Let's look at the structure of one product entry
products[0]

In [ ]:
# Extract product info: name, description, and PDF link
product_data = []
for product in products:
    info = {}
    
    # Get the product name
    title = product.find("div", attrs={"class": "ds-title"})
    info["name"] = title.string.strip() if title else None
    
    # Get the description
    desc = product.find("div", attrs={"class": "ds-desc"})
    if desc:
        p_tag = desc.find("p")
        info["description"] = p_tag.string.strip() if p_tag and p_tag.string else None
    
    # Get the PDF download link
    link = product.find("a", attrs={"class": "ds-download"})
    if link:
        info["pdf_url"] = "https://www.celgard.com" + link.attrs["href"]
    
    product_data.append(info)

df_celgard = pd.DataFrame(product_data)
df_celgard

In [ ]:
# Also extract the category each product belongs to
# Categories are in <h3> tags inside <div class="datasheets"> containers
categories = soup.find_all("div", attrs={"class": "datasheets"})

product_categories = []
for category_div in categories:
    heading = category_div.find("h3")
    if heading:
        category_name = heading.string.strip() if heading.string else heading.get_text().strip()
        items = category_div.find_all("div", attrs={"class": "ds-item"})
        for item in items:
            title = item.find("div", attrs={"class": "ds-title"})
            if title:
                product_categories.append({
                    "name": title.string.strip(),
                    "category": category_name
                })

df_categories = pd.DataFrame(product_categories)
df_celgard = df_celgard.merge(df_categories, on="name", how="left")
df_celgard

Now we can see all 30 Celgard products organized by category (Monolayer PP, Trilayer PP/PE/PP, Ceramic Coated, etc.), each with a link to its PDF datasheet.

### 1b. Loading the OpenPoly Polymer Database

OpenPoly is a dataset of 741 polymers with 26 experimentally measured properties. This will serve as our screening pool — polymers that have never been tested as battery separators.

In [ ]:
df_polymers = pd.read_csv("data/final_polymer_properties_fromliterature.csv")
print(f"Shape: {df_polymers.shape}")
df_polymers.head()

In [ ]:
# Properties relevant to battery separators
separator_relevant = [
    "Name", "Tg (K)", "Tm (K)", "Td (K)",
    "Tensile Strength (MPa)", "Young's Modulus (MPa)",
    "Elongation at Break (%)", "Thermal Conductivity (W/m K)",
    "Crystallization Temperature (K)", "Water Contact Angle (deg)",
    "Water Uptake (%)", "Hardness (MPa)", "Compressive Strength (MPa)"
]

df_relevant = df_polymers[separator_relevant]
df_relevant

In [ ]:
# How much data do we have for each property?
df_relevant.drop(columns="Name").notna().sum().sort_values(ascending=False)

In [ ]:
# Focus on the best-covered properties for modeling
# These are the features we can use for both training and screening
key_features = ["Tg (K)", "Tm (K)", "Td (K)",
                "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                "Elongation at Break (%)"]

# How many polymers have ALL of these properties?
df_complete = df_relevant.dropna(subset=key_features)
print(f"Polymers with all key properties: {len(df_complete)}")
df_complete

In [ ]:
# Let's look at the known separator polymers in this dataset
known_separator_names = ["PE", "PP", "PVDF", "PAN", "HDPE", "LDPE",
                         "polypropylene", "polyethylene", 
                         "poly(vinylidene fluoride)", "polyacrylonitrile",
                         "cellulose", "PLA", "PET", "nylon", "PI",
                         "polyimide", "PTFE", "PVA"]

# Search for these in the Name column (case-insensitive)
mask = df_polymers["Name"].str.lower().isin([n.lower() for n in known_separator_names])
df_known = df_polymers[mask]
print(f"Found {len(df_known)} known separator-relevant polymers in OpenPoly")
df_known[["Name"] + key_features]

### 1c. Exploring the Data

Let's visualize the property landscape of all polymers in the database.

In [ ]:
import numpy as np

# Scatter plot: Tg vs Tm for all polymers
ax = df_polymers.plot.scatter(x="Tg (K)", y="Tm (K)", alpha=0.4, 
                              color="gray", label="All polymers")

# Highlight known separator polymers
df_known.plot.scatter(x="Tg (K)", y="Tm (K)", alpha=0.9, 
                      color="red", label="Known separator polymers", ax=ax)

ax.set_title("Thermal Properties of Polymers")
ax.legend()

In [ ]:
# Scatter plot: Tensile Strength vs Young's Modulus
ax = df_polymers.plot.scatter(x="Young's Modulus (MPa)", 
                              y="Tensile Strength (MPa)", 
                              alpha=0.4, color="gray",
                              label="All polymers")

df_known.plot.scatter(x="Young's Modulus (MPa)", 
                      y="Tensile Strength (MPa)", 
                      alpha=0.9, color="red",
                      label="Known separator polymers", ax=ax)

ax.set_title("Mechanical Properties of Polymers")
ax.legend()

## Next Steps

1. Parse Celgard PDF datasheets for separator-specific properties (porosity, Gurley, thermal shrinkage)
2. Compile additional separator data from review papers (Nature Energy 2019, NASA reports)
3. Define safety labels based on thermal shrinkage thresholds
4. Build KNN classifier pipeline
5. Screen OpenPoly polymers with trained model